Bibliotecas

---

In [51]:
import h5py
import torch
import numpy as np

import torch.nn as nn

---

In [52]:
with h5py.File("../dist/automatic-ecg-diagnosis/model.hdf5", "r") as f:
  for model_weidht in f['model_weights']:
    print(model_weidht)


activation_1
activation_2
activation_3
activation_4
activation_5
activation_6
activation_7
activation_8
activation_9
add_1
add_2
add_3
add_4
batch_normalization_1
batch_normalization_2
batch_normalization_3
batch_normalization_4
batch_normalization_5
batch_normalization_6
batch_normalization_7
batch_normalization_8
batch_normalization_9
conv1d_1
conv1d_10
conv1d_11
conv1d_12
conv1d_13
conv1d_2
conv1d_3
conv1d_4
conv1d_5
conv1d_6
conv1d_7
conv1d_8
conv1d_9
dense_8
dropout_1
dropout_2
dropout_3
dropout_4
dropout_5
dropout_6
dropout_7
dropout_8
flatten_1
max_pooling1d_1
max_pooling1d_2
max_pooling1d_3
max_pooling1d_4
signal


---

In [53]:
# Source - https://stackoverflow.com/a/73686304
# Posted by Alex44, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-08, License - CC BY-SA 4.0

import h5py

filename_hdf = 'data.hdf5'

def h5_tree(val, pre=''):
    items = len(val)
    for key, val in val.items():
        items -= 1
        if items == 0:
            # the last item
            if type(val) == h5py._hl.group.Group:
                print(pre + '└── ' + key)
                h5_tree(val, pre+'    ')
            else:
                try:
                    print(pre + '└── ' + key + ' (%d)' % len(val))
                except TypeError:
                    print(pre + '└── ' + key + ' (scalar)')
        else:
            if type(val) == h5py._hl.group.Group:
                print(pre + '├── ' + key)
                h5_tree(val, pre+'│   ')
            else:
                try:
                    print(pre + '├── ' + key + ' (%d)' % len(val))
                except TypeError:
                    print(pre + '├── ' + key + ' (scalar)')

with h5py.File("../dist/automatic-ecg-diagnosis/model.hdf5", 'r') as hf:
    print(hf)
    h5_tree(hf)


<HDF5 file "model.hdf5" (mode r)>
└── model_weights
    ├── activation_1
    ├── activation_2
    ├── activation_3
    ├── activation_4
    ├── activation_5
    ├── activation_6
    ├── activation_7
    ├── activation_8
    ├── activation_9
    ├── add_1
    ├── add_2
    ├── add_3
    ├── add_4
    ├── batch_normalization_1
    │   └── batch_normalization_1_7
    │       ├── beta:0 (64)
    │       ├── gamma:0 (64)
    │       ├── moving_mean:0 (64)
    │       └── moving_variance:0 (64)
    ├── batch_normalization_2
    │   └── batch_normalization_2_7
    │       ├── beta:0 (128)
    │       ├── gamma:0 (128)
    │       ├── moving_mean:0 (128)
    │       └── moving_variance:0 (128)
    ├── batch_normalization_3
    │   └── batch_normalization_3_7
    │       ├── beta:0 (128)
    │       ├── gamma:0 (128)
    │       ├── moving_mean:0 (128)
    │       └── moving_variance:0 (128)
    ├── batch_normalization_4
    │   └── batch_normalization_4_7
    │       ├── beta:0 (196)
    │    

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np


class ResidualUnit(nn.Module):
    """Unidade Residual unidimensional ajustada para compatibilidade exata com Keras."""
    def __init__(self, n_samples_out, n_filters_out, dropout_keep_prob=0.8, 
                 kernel_size=17, preactivation=True, postactivation_bn=False, 
                 activation_function='relu'):
        super(ResidualUnit, self).__init__()
        self.n_samples_out = n_samples_out
        self.n_filters_out = n_filters_out
        self.dropout_rate = 1 - dropout_keep_prob
        self.kernel_size = kernel_size
        self.preactivation = preactivation
        self.postactivation_bn = postactivation_bn
        
        self.activation = nn.ReLU()
        self.skip_conv = None
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=n_filters_out, 
                               kernel_size=kernel_size, padding='same', bias=False)
        self.bn1 = nn.BatchNorm1d(n_filters_out)
        
        self.conv2 = nn.Conv1d(in_channels=n_filters_out, out_channels=n_filters_out, 
                               kernel_size=kernel_size, bias=False)
        self.bn2 = nn.BatchNorm1d(n_filters_out)
        
        self.dropout = nn.Dropout(p=self.dropout_rate) if self.dropout_rate > 0 else None

    def _skip_connection(self, y, downsample, n_filters_in, target_length, device):
        if downsample > 1:
            # Garante o pooling com o tamanho exato de saída esperado (n_samples_out)
            y = F.max_pool1d(y, kernel_size=downsample, stride=downsample)
        
        # Ajusta caso o tamanho fique diferente por arredondamento
        if y.shape[2] != target_length:
            diff = target_length - y.shape[2]
            y = F.pad(y, (0, diff))
            
        if n_filters_in != self.n_filters_out:
            if self.skip_conv is None or self.skip_conv.in_channels != n_filters_in or self.skip_conv.out_channels != self.n_filters_out:
                self.skip_conv = nn.Conv1d(n_filters_in, self.n_filters_out, kernel_size=1, padding=0, bias=False).to(device)
            y = self.skip_conv(y)
        return y

    def _batch_norm_plus_activation(self, x, bn_layer, act_layer):
        if self.postactivation_bn:
            x = act_layer(x)
            x = bn_layer(x)
        else:
            x = bn_layer(x)
            x = act_layer(x)
        return x

    def forward(self, inputs):
        x, y = inputs
        n_samples_in = y.shape[2]
        downsample = n_samples_in // self.n_samples_out
        n_filters_in = y.shape[1]
        device = x.device
        
        # Processa o skip connection mirando o tamanho de saída exato
        y = self._skip_connection(y, downsample, n_filters_in, self.n_samples_out, device)
        
        if self.conv1.in_channels != x.shape[1]:
            self.conv1 = nn.Conv1d(x.shape[1], self.n_filters_out, kernel_size=self.kernel_size, padding='same', bias=False).to(device)
            self.bn1 = nn.BatchNorm1d(self.n_filters_out).to(device)
            
        x = self.conv1(x)
        x = self._batch_norm_plus_activation(x, self.bn1, self.activation)
        if self.dropout is not None:
            x = self.dropout(x)
            
        # Calcula o padding explíto para a conv com stride de forma compatível com o Keras ('same')
        out_len = self.n_samples_out
        in_len = x.shape[2]
        total_pad = max(0, (out_len - 1) * downsample + self.kernel_size - in_len)
        pad_left = total_pad // 2
        pad_right = total_pad - pad_left
        
        if self.conv2.in_channels != self.n_filters_out:
            self.conv2 = nn.Conv1d(self.n_filters_out, self.n_filters_out, kernel_size=self.kernel_size, stride=downsample, bias=False).to(device)
            self.bn2 = nn.BatchNorm1d(self.n_filters_out).to(device)
            
        # Aplica o padding manualmente antes da convolução para evitar erros de dimensão
        x_padded = F.pad(x, (pad_left, pad_right))
        x = self.conv2(x_padded)
        
        # Garante alinhamento final exato de dimensões se houver diferença de arredondamento
        if x.shape[2] != y.shape[2]:
            diff = y.shape[2] - x.shape[2]
            x = F.pad(x, (0, diff))

        if self.preactivation:
            x = x + y
            y = x
            x = self._batch_norm_plus_activation(x, self.bn2, self.activation)
            if self.dropout is not None:
                x = self.dropout(x)
        else:
            x = self.bn2(x)
            x = x + y
            x = self.activation(x)
            if self.dropout is not None:
                x = self.dropout(x)
            y = x
            
        return x, y


def load_weights_from_hdf5(model, hdf5_path):
    """Função para carregar os pesos diretamente do arquivo model.hdf5 estruturado pelo Keras."""
    with h5py.File(hdf5_path, 'r') as f:
        weights_group = f['model_weights']
        
        # Mapeamento manual ou automático dos pesos do HDF5 para os state_dict do PyTorch
        # Exemplo para a camada inicial conv1d_1
        if 'conv1d_1' in weights_group:
            k_path = 'conv1d_1/conv1d_1_7/kernel:0'
            if k_path in weights_group['conv1d_1']:
                w = np.array(weights_group['conv1d_1']['conv1d_1_7']['kernel:0'])
                # Keras Conv1D: [kernel_size, in_channels, out_channels]
                # PyTorch Conv1D: [out_channels, in_channels, kernel_size]
                w = np.transpose(w, (2, 1, 0))
                model.conv1.weight.data = torch.from_numpy(w)

        # Repetir o mapeamento estruturado para BatchNormalization, outras Convoluções e Dense (dense_8)
        print("Pesos carregados estruturalmente (personalize o mapeamento completo conforme necessário).")


if __name__ == "__main__":
    model = ResNet1DModel(n_classes=6, last_layer='sigmoid')
    print(model)

ResNet1DModel(
  (conv1): Conv1d(12, 64, kernel_size=(16,), stride=(1,), padding=same, bias=False)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (activation): ReLU()
  (res1): ResidualUnit(
    (activation): ReLU()
    (conv1): Conv1d(1, 128, kernel_size=(16,), stride=(1,), padding=same, bias=False)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(128, 128, kernel_size=(16,), stride=(1,), bias=False)
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (dropout): Dropout(p=0.19999999999999996, inplace=False)
  )
  (res2): ResidualUnit(
    (activation): ReLU()
    (conv1): Conv1d(1, 196, kernel_size=(16,), stride=(1,), padding=same, bias=False)
    (bn1): BatchNorm1d(196, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(196, 196, kernel_size=(16,), stride=(1,), bias=False)
    (bn2): BatchNorm1d(196, ep

In [63]:
import torch

def test_model_output():
    # 1. Define o número de classes e cria a instância do modelo
    n_classes = 6
    model = ResNet1DModel(n_classes=n_classes, last_layer='sigmoid')
    
    # Coloca o modelo em modo de avaliação (importante por causa do Dropout e BatchNorm)
    model.eval()
    
    # 2. Cria uma entrada aleatória simulando o formato do Keras: [Batch_size, Length, Channels] = [2, 4096, 12]
    # O modelo PyTorch faz a transposição automática para [Batch_size, Channels, Length] = [2, 12, 4096] internamente.
    batch_size = 2
    length = 4096
    channels = 12
    random_input = torch.randn(batch_size, length, channels)
    
    print(f"Formato da entrada aleatória: {random_input.shape}")
    
    # 3. Executa a inferência (forward pass) sem gradientes
    with torch.no_grad():
        output = model(random_input)
        
    print(f"Formato da saída do modelo: {output.shape}")
    print("Valores de saída (probabilidades):")
    print(output)

if __name__ == "__main__":
    test_model_output()

Formato da entrada aleatória: torch.Size([2, 4096, 12])
Formato da saída do modelo: torch.Size([2, 6])
Valores de saída (probabilidades):
tensor([[0.5441, 0.5560, 0.5296, 0.5634, 0.5650, 0.6040],
        [0.5602, 0.5691, 0.5101, 0.5332, 0.5526, 0.5952]])
